In [ ]:
import os

from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login

login(os.environ["HF_TOKEN"])

from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B")
model

In [ ]:
model_name = "google/vaultgemma-1b"
models = ["gpt2", "google/gemma-2b", "EleutherAI/pythia-70m", "meta-llama/Llama-2-7b-hf", "EleutherAI/gpt-neo-125M"]
for model_name in models:
    print(model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- Disable flash attention (important for higher-order grads) ---
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)

device = "cuda" if torch.cuda.is_available() else "cpu"

# ================================
# CONFIG
# ================================
USE_LAMP = True          # False → TAG, True → LAMP
LAMBDA_LM = 1.0
LR = 0.01
STEPS = 600

model_name = "gpt2"

# ================================
# MODEL
# ================================
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    attn_implementation="eager"
).to(device)

model.eval()

# ================================
# TEST DATA
# ================================
texts = [
    "My name is John Snow",
    "The cat is sitting on the table",
    "I love machine learning",
    "This is a privacy attack test"
]

embedding_layer = model.get_input_embeddings()
def initialize_dummy_embeddings(
    model,
    tokenizer,
    lm_model,
    lm_tokenizer,
    seq_len,
    batch_size=1,
    method="random",          # "random" or "lm"
    num_candidates=5,
    device="cpu",
):
    """
    Returns: dummy_embeds (requires_grad=True)
    """

    embedding_layer = model.get_input_embeddings()
    hidden_dim = embedding_layer.embedding_dim

    # ================================
    # RANDOM INIT
    # ================================
    if method == "random":
        dummy_embeds = torch.randn(
            (batch_size, seq_len, hidden_dim),
            device=device,
            requires_grad=True
        )
        return dummy_embeds

    # ================================
    # LM INIT
    # ================================
    elif method == "lm":

        # start token
        input_ids = lm_tokenizer.encode("the", return_tensors="pt").to(device)

        # generate candidate sequences
        outputs = lm_model.generate(
            input_ids,
            max_length=seq_len + 5,
            num_return_sequences=num_candidates,
            do_sample=True,
            no_repeat_ngram_size=2
        )

        # remove prefix
        outputs = outputs[:, -seq_len:]

        # convert to embeddings
        candidates = embedding_layer(outputs)

        # pick best based on LM loss (fluency)
        best_embed = None
        best_loss = float("inf")

        for i in range(num_candidates):
            candidate_ids = outputs[i].unsqueeze(0)

            with torch.no_grad():
                out = lm_model(candidate_ids, labels=candidate_ids)
                loss = out.loss.item()

            if loss < best_loss:
                best_loss = loss
                best_embed = candidates[i].unsqueeze(0)

        dummy_embeds = best_embed.clone().detach().requires_grad_(True)
        return dummy_embeds

    else:
        raise ValueError("method must be 'random' or 'lm'")
        
for text in texts:

    print("\n============================")
    print("TEXT:", text)

    inputs = tokenizer(text, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]

    # ================================
    # TRUE GRADIENTS
    # ================================
    params = [p for p in model.parameters() if p.requires_grad]
    attention_mask = torch.ones_like(input_ids)
    outputs = model(input_ids=input_ids, labels=input_ids, attention_mask=attention_mask)
    loss = outputs.loss

    true_grads = torch.autograd.grad(loss, params)
    true_grads = [g.detach().clone() for g in true_grads]

    # ================================
    # INIT DUMMY EMBEDS
    # ================================
    #dummy_embeds = ( embedding_layer(input_ids).clone().detach() + 0.1 * torch.randn_like(embedding_layer(input_ids))).detach().requires_grad_(True)
    dummy_embeds = initialize_dummy_embeddings(model=model, tokenizer=tokenizer, lm_model=model, lm_tokenizer=tokenizer, 
                                               seq_len=input_ids.shape[1], method="lm", device=device)
    optimizer = torch.optim.Adam([dummy_embeds], lr=LR)

    # ================================
    # OPTIMIZATION (TAG / LAMP)
    # ================================
    for step in range(STEPS):

        optimizer.zero_grad()
        model.zero_grad()

        attention_mask = torch.ones_like(input_ids)

        outputs = model(inputs_embeds=dummy_embeds, labels=input_ids, attention_mask=attention_mask)

        dummy_loss = outputs.loss

        dummy_grads = torch.autograd.grad(
            dummy_loss,
            params,
            create_graph=True,
            allow_unused=True
        )

        grad_loss = 0

        for g1, g2 in zip(dummy_grads, true_grads):
            if g1 is None or g2 is None:
                continue

            g1 = g1 / (g1.norm() + 1e-8)
            g2 = g2 / (g2.norm() + 1e-8)

            grad_loss += ((g1 - g2) ** 2).sum()

        if USE_LAMP:
            total_loss = grad_loss + LAMBDA_LM * dummy_loss
        else:
            total_loss = grad_loss

        total_loss.backward()
        optimizer.step()

        dummy_embeds.data = dummy_embeds.data.clamp(-1, 1)

        if step % 100 == 0:
            print(f"Step {step}, Grad Loss: {grad_loss.item():.4f}")

    # ================================
    # DECODE TAG/LAMP
    # ================================
    with torch.no_grad():
        logits = torch.matmul(dummy_embeds, embedding_layer.weight.T)
        predicted_ids = torch.argmax(logits, dim=-1)

    reconstructed = tokenizer.decode(predicted_ids[0], skip_special_tokens=True)

    # ================================
    # DAGER-LITE (no optimization)
    # ================================
    with torch.no_grad():
        embed_matrix = embedding_layer.weight  # [vocab, dim]

        # crude gradient signal (just first param)
        grad_signal = true_grads[0].mean(dim=0)

        scores = torch.matmul(embed_matrix, grad_signal)

        top_tokens = torch.topk(scores, k=input_ids.shape[1], dim=0).indices
        dager_ids = top_tokens.unsqueeze(0)

        dager_text = tokenizer.decode(dager_ids[0], skip_special_tokens=True)

    # ================================
    # HYBRID (LM-guided selection)
    # ================================
    with torch.no_grad():
        logits = torch.matmul(dummy_embeds, embedding_layer.weight.T)

        topk = torch.topk(logits, k=5, dim=-1).indices

        best_seq = None
        best_loss = float("inf")

        for k in range(5):
            candidate = topk[:, :, k]

            out = model(candidate, labels=candidate)
            loss = out.loss.item()

            if loss < best_loss:
                best_loss = loss
                best_seq = candidate

        hybrid_text = tokenizer.decode(best_seq[0], skip_special_tokens=True)

    # ================================
    # RESULTS
    # ================================
    print("\nGround Truth:", text)
    print("TAG/LAMP:", reconstructed)
    print("DAGER-lite:", dager_text)
    print("Hybrid:", hybrid_text)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


TEXT: My name is John Snow


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Step 0, Grad Loss: 215.3853
Step 100, Grad Loss: 80.1386
Step 200, Grad Loss: 54.2341
Step 300, Grad Loss: 48.2659
Step 400, Grad Loss: 45.5943
Step 500, Grad Loss: 42.2456

Ground Truth: My name is John Snow
TAG/LAMP: My
 nameJohn is
DAGER-lite: heimer Dengedintufu
Hybrid: My
 nameJohn is

TEXT: The cat is sitting on the table


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Step 0, Grad Loss: 235.3716
Step 100, Grad Loss: 71.7500
Step 200, Grad Loss: 42.2568
Step 300, Grad Loss: 33.1408
Step 400, Grad Loss: 27.0874
Step 500, Grad Loss: 23.3777

Ground Truth: The cat is sitting on the table
TAG/LAMP: The cat is cat horizont the've
DAGER-lite:  Rangers Sea RangerSea juvenile juveniles Marine
Hybrid: The cat is cat horizont the've

TEXT: I love machine learning


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Step 0, Grad Loss: 222.7784
Step 100, Grad Loss: 103.0721
Step 200, Grad Loss: 76.7601
Step 300, Grad Loss: 66.2531
Step 400, Grad Loss: 67.5325
Step 500, Grad Loss: 64.6140


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Ground Truth: I love machine learning
TAG/LAMP: I corridmachine

DAGER-lite:  barrel tub dope rim
Hybrid: MyMachinecius.

TEXT: This is a privacy attack test
Step 0, Grad Loss: 260.3939
Step 100, Grad Loss: 95.5691
Step 200, Grad Loss: 74.1326
Step 300, Grad Loss: 36.6661
Step 400, Grad Loss: 31.9376
Step 500, Grad Loss: 29.7390

Ground Truth: This is a privacy attack test
TAG/LAMP: Thiscome- privacy attacksa
DAGER-lite: UGHverningazeera excuse Reck Laugh
Hybrid: Thiscome- privacy attacksa


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- Disable flash attention (needed for higher-order grads) ---
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)

device = "cuda" if torch.cuda.is_available() else "cpu"

# ================================
# CONFIG
# ================================
USE_LAMP = True          # False → TAG, True → LAMP
LAMBDA_LM = 0.1          # strength of LM prior
LR = 0.05
STEPS = 200

# ================================
# MODEL
# ================================
 
models = ["gpt2", "EleutherAI/pythia-70m", "google/gemma-2b"]
#model_name = "google/vaultgemma-1b", "meta-llama/Llama-2-7b-hf"
for model_name in models:
    print(f"Model name {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    model = AutoModelForCausalLM.from_pretrained(model_name, attn_implementation="eager").to(device)
    
    model.eval()
    
    # ================================
    # GROUND TRUTH
    # ================================
    text = "My name is John Snow"
    inputs = tokenizer(text, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]
    
    # ================================
    # TRUE GRADIENTS
    # ================================
    params = [p for p in model.parameters() if p.requires_grad]
    
    outputs = model(input_ids, labels=input_ids)
    loss = outputs.loss
    
    true_grads = torch.autograd.grad(loss, params)
    true_grads = [g.detach().clone() for g in true_grads]
    
    # ================================
    # INITIALIZE DUMMY
    # ================================
    embedding_layer = model.get_input_embeddings()
    seq_len = input_ids.shape[1]
    hidden_dim = embedding_layer.embedding_dim
    
    dummy_embeds = (
        embedding_layer(input_ids).clone().detach()
        + 0.1 * torch.randn_like(embedding_layer(input_ids))
    ).detach().requires_grad_(True)
    #dummy_embeds.requires_grad = True
    
    #dummy_embeds = torch.randn((1, seq_len, hidden_dim), device=device, requires_grad=True)
    
    optimizer = torch.optim.Adam([dummy_embeds], lr=LR)
    
    # ================================
    # OPTIMIZATION LOOP
    # ================================
    for step in range(STEPS):
    
        optimizer.zero_grad()
        model.zero_grad()
    
        outputs = model(inputs_embeds=dummy_embeds, labels=input_ids)
        dummy_loss = outputs.loss   # LM loss (used in LAMP)
    
        dummy_grads = torch.autograd.grad(
            dummy_loss,
            params,
            create_graph=True,
            allow_unused=True
        )
    
        grad_loss = 0
    
        for g1, g2 in zip(dummy_grads, true_grads):
            if g1 is None or g2 is None:
                continue
    
            # normalize (important)
            g1 = g1 / (g1.norm() + 1e-8)
            g2 = g2 / (g2.norm() + 1e-8)
    
            grad_loss += ((g1 - g2) ** 2).sum()
    
        # ================================
        # SWITCH: TAG vs LAMP
        # ================================
        if USE_LAMP:
            total_loss = grad_loss + LAMBDA_LM * dummy_loss
        else:
            total_loss = grad_loss
    
        total_loss.backward()
        optimizer.step()
    
        # stabilize
        dummy_embeds.data = dummy_embeds.data.clamp(-1, 1)
    
        if step % 50 == 0:
            print(f"Step {step}, Grad Loss: {grad_loss.item():.4f}, Total: {total_loss.item():.4f}")
    
    # ================================
    # DECODE
    # ================================
    with torch.no_grad():
        logits = torch.matmul(dummy_embeds, embedding_layer.weight.T)
        predicted_ids = torch.argmax(logits, dim=-1)
    
    reconstructed = tokenizer.decode(predicted_ids[0], skip_special_tokens=True)
    
    print("\nGround Truth:", text)
    print("Reconstructed:", reconstructed)

Model name gpt2


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step 0, Grad Loss: 20.5107, Total: 21.0072
Step 50, Grad Loss: 3.0735, Total: 3.5455
Step 100, Grad Loss: 0.9909, Total: 1.4626
Step 150, Grad Loss: 0.6842, Total: 1.1564

Ground Truth: My name is John Snow
Reconstructed: My name is John Snow
Model name google/gemma-2b


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]